<a href="https://colab.research.google.com/github/arpitelias/ArpitJoshuaElias_FlyRank/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arpitelias/ArpitJoshuaElias_FlyRank/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [7]:
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and not os.path.isdir("flyrank-ml-internship-starter"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/flyrank-bih/flyrank-ml-internship-starter", "flyrank-ml-internship-starter"], check=True)
if IN_COLAB:
    os.chdir("/content/flyrank-ml-internship-starter")
import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"{len(df):,} pages, {len(df.columns)} columns")

30,000 pages, 44 columns


## 1. My lane as an ML task (type)

Scoring, specifically ranking.

The question is which pages someone should look at first, not whether a page will decline. That's an ordering problem. The output is a score per page and only the top of the list matters, so precision near the top is what counts, not accuracy across all 30,000 rows.

I looked at classification and dropped it. There's no yes/no label in the data, so I'd have to invent one from a threshold I picked myself, and the model would just learn my threshold back. Notebook 02 did exactly that: a tree given a rule-derived label found the cutoff and scored a perfect 1.000. That's not skill, it's circular.

Clustering doesn't fit either. I already know what I'm looking for.

In [8]:
print(f"rows: {len(df):,}")
print(f"one row per content_id: {df['content_id'].is_unique}")
print(f"ready-made label columns: {[c for c in df.columns if 'label' in c.lower() or 'target' in c.lower() or 'flag' in c.lower()]}")
print(f"reviewer capacity 20/week covers top {20 / len(df):.2%} of the list")

rows: 30,000
one row per content_id: True
ready-made label columns: []
reviewer capacity 20/week covers top 0.07% of the list


## 2. Target or proxy

Target: actual CTR. The thing I rank on is the gap between actual and predicted.

CTR is observed. It's clicks over impressions, both measured, nobody's judgement involved. So I train a model to predict a page's CTR from signals that aren't CTR itself, mainly position, word count and metadata. Then I rank pages by actual minus predicted.

The reason for the extra step: I want the gap to come out of the model, not out of a threshold I picked. If I defined "underperforming" as CTR below some number I chose, I'd be back in notebook 02 territory, teaching a model my own rule and watching it hand the rule back. Here the target is real and the gap is a residual.

Worth being clear this is a proxy. A large negative residual means the page gets fewer clicks than similar pages at similar positions. It doesn't mean the title is bad. It means worth a look.

In [9]:
print(df[["ctr", "impressions_90d", "clicks_90d"]].describe().round(2).to_string())
recomputed = (df["clicks_90d"] / df["impressions_90d"] * 100).round(3)
print(f"ctr matches clicks/impressions as a percentage: {np.allclose(recomputed, df['ctr'], atol=0.01)}")
print(f"ctr null count: {df['ctr'].isna().sum()}")


            ctr  impressions_90d  clicks_90d
count  30000.00         30000.00    30000.00
mean       0.51          5200.37       16.10
std        3.28         16838.02       75.08
min        0.00             1.00        0.00
25%        0.00            81.00        0.00
50%        0.07           731.00        1.00
75%        0.29          3615.25        7.00
max      100.00        517715.00     4178.00
ctr matches clicks/impressions as a percentage: True
ctr null count: 0


## 3. Success metric
Precision@50, judged against the base rate.

Fifty because that's roughly what a reviewer can work through, and it's the number the starter pipeline already uses, so I can compare against a baseline rather than inventing a scale.

What counts as good: the top 50 by residual has to contain more genuinely underperforming pages than 50 pages picked at random would. That's the floor, and it's a low bar I need to clear before anything else matters. The starter's hand rule dropped to 0.580 against a base rate of 0.542 on held-out data in notebook 02, so a rule that looked fine in-sample was close to random once tested honestly.

One thing I can't do yet. There's no observed label for "this page was actually worth reviewing," so precision here is measured against a proxy, not against a reviewer's verdict. I'll say directional rather than claiming the queue is correct.

In [10]:
print(f"pages with impressions < 100: {(df['impressions_90d'] < 100).mean():.1%}")
tiny = df[df["impressions_90d"] < 10]
print(f"pages under 10 impressions: {len(tiny):,}, of which ctr == 100: {(tiny['ctr'] == 100).sum()}")
print(f"max ctr overall: {df['ctr'].max():.1f}% on {df.loc[df['ctr'].idxmax(), 'impressions_90d']:.0f} impressions")
eligible = df[df["impressions_90d"] >= 100]
print(f"after a 100-impression floor: {len(eligible):,} pages, top 50 is {50/len(eligible):.2%}")


pages with impressions < 100: 26.6%
pages under 10 impressions: 3,746, of which ctr == 100: 11
max ctr overall: 100.0% on 1 impressions
after a 100-impression floor: 22,006 pages, top 50 is 0.23%


## 4. The unit of analysis, as a real dataframe

One row = one page, on one client's site, over a fixed 90-day window.

content_id is unique across all 30,000 rows, so there's no stacking and no repeated measures. It's a cross-section, one snapshot, not a time series.

I'm scoring 22,006 of those rows, the ones with at least 100 impressions in the window. The floor isn't a tidying step, it's load-bearing. Eleven pages in this data show a perfect 100% CTR off a single impression. Without a floor those sit at the top of any CTR ranking and the queue is worthless on day one. 100 is a judgement call and I'd want to test it, but it has to be somewhere.

In [11]:
scope = df[df["impressions_90d"] >= 100].copy()
print(f"unit: one row per page. rows in scope: {len(scope):,} of {len(df):,}")
print(f"clients represented: {scope['client_id'].nunique()} of {df['client_id'].nunique()}")
print(scope[["content_id", "client_id", "position_tier", "avg_position", "impressions_90d", "clicks_90d", "ctr", "word_count"]].head(5).to_string(index=False))
print(f"pages per client in scope: median {scope['client_id'].value_counts().median():.0f}, max {scope['client_id'].value_counts().max():,}")


unit: one row per page. rows in scope: 22,006 of 30,000
clients represented: 30 of 32
          content_id         client_id position_tier  avg_position  impressions_90d  clicks_90d  ctr  word_count
content_304f48230142 client_f369cb89fc      striking          10.6             3803          29 0.76      3221.0
content_a1fb4e703a9e client_4e07408562      page_3_5          20.3            15320           7 0.05      2481.0
content_9aa793d4d895 client_7f2253d7e2      page_3_5          36.5            12581          11 0.09      3515.0
content_331d6c4de07b client_19581e27de        page_1           6.2            11751          58 0.49         NaN
content_d99b7a2d90ca client_3fdba35f04      page_3_5          44.0            19140          24 0.13      2803.0
pages per client in scope: median 254, max 6,579


## 5. Why ML beats a fixed rule here

Because the sensible if-statement doesn't work, and I checked.

The obvious rule is "flag pages with low CTR." That fails immediately: CTR falls off with position, so the rule just returns whatever sits lowest on the page. The next rule is "flag low CTR within a position band," which is closer, but the bands in this data are unreliable. A page tagged striking sits at position 10.6 while a page tagged page_3_5 sits at 36.5, so the labels don't mean what they say.

Underneath that, the relationship isn't a single line. Position matters most but not evenly, word count interacts with it, and 26.6% of pages have too little volume for CTR to mean anything at all. A fixed rule has to pick one cutoff and apply it everywhere. A model can hold several signals at once and produce an expected CTR per page, which is what the residual needs.

I'd rather this be tested than asserted. The check below is whether one signal on its own is enough to explain CTR. If it were, the rule would be fine and I shouldn't be building a model.

What the check showed. Median CTR falls cleanly across position deciles, 0.29 down to 0.00, no reversals. So position really does drive clicks. But the correlation is only -0.239, so position explains a small share of the variation. Direction is strong, magnitude is weak, and that gap is the case for a model: the pattern is real but one variable barely captures it. Word count on its own is nearly flat at -0.036.

One problem I'm not going to hide: word_count is missing on 29.8% of pages in scope. If I use it as a feature I need a strategy for that, and I'd want to check whether the missingness itself is informative before filling it in.

In [12]:
scope = df[df["impressions_90d"] >= 100].copy()
print("correlation with ctr:")
for c in ["avg_position", "word_count", "impressions_90d"]:
    print(f"  {c}: {scope['ctr'].corr(scope[c]):.3f}")
print()
print("median ctr by position decile:")
scope["pos_decile"] = pd.qcut(scope["avg_position"], 10, labels=False, duplicates="drop")
print(scope.groupby("pos_decile")["ctr"].median().round(3).to_string())
print()
print(f"word_count missing in scope: {scope['word_count'].isna().sum():,} ({scope['word_count'].isna().mean():.1%})")


correlation with ctr:
  avg_position: -0.239
  word_count: -0.036
  impressions_90d: 0.047

median ctr by position decile:
pos_decile
0    0.29
1    0.25
2    0.21
3    0.17
4    0.16
5    0.15
6    0.14
7    0.10
8    0.06
9    0.00

word_count missing in scope: 6,555 (29.8%)


## Self-check

Before you submit, confirm each line honestly:

- [ ✅] Every section above is filled — markdown thinking AND the code that backs it
- [✅ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ✅] No client names, URLs, or private queries anywhere
- [ ✅] My claims use careful words: observed, measured, directional, decision-support
- [✅ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.